# YOLO11 Fall/Non-Fall Pose 학습 (Google Colab)

Caterpillar Roboflow v3 데이터셋(12관절, `Fall`/`Non-Fall`)을 다운로드하고 YOLO11n Pose를 학습합니다.

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤 위에서부터 순서대로 실행하세요.

In [ ]:
# 1. GPU와 런타임 확인
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 유형을 T4 GPU로 변경하세요.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. 필요한 패키지 설치
%pip install -q -U ultralytics pyyaml

## Roboflow API 키 준비

권장 방식은 Colab 왼쪽의 **Secrets(열쇠 아이콘)** 에 `ROBOFLOW_API_KEY`를 추가하는 것입니다.
Secrets에 없으면 다음 셀에서 화면에 표시되지 않는 숨김 입력으로 받습니다.

In [ ]:
# 3. API 키를 노출하지 않고 읽기
from getpass import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass('Roboflow Private API Key: ')
if not ROBOFLOW_API_KEY:
    raise ValueError('API 키가 비어 있습니다.')
print('API key loaded:', bool(ROBOFLOW_API_KEY))

In [ ]:
# 4. Roboflow YOLOv11 Pose 데이터셋 다운로드
import json
import shutil
import time
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

WORKDIR = Path('/content/fall_pose')
DATASET_DIR = WORKDIR / 'dataset'
RUNS_DIR = WORKDIR / 'runs'
WORKDIR.mkdir(parents=True, exist_ok=True)

endpoint = (
    'https://api.roboflow.com/caterpillar-dlcux/'
    'fall-detection-apsso/3/yolov11'
)
url = endpoint + '?' + urllib.parse.urlencode({'api_key': ROBOFLOW_API_KEY})
export_link = None
for attempt in range(60):
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = json.load(response)
    export = payload.get('export') or {}
    export_link = export.get('link') if isinstance(export, dict) else export
    if export_link:
        break
    if payload.get('error'):
        raise RuntimeError(str(payload['error']).replace(ROBOFLOW_API_KEY, '***'))
    if attempt == 0:
        print('Roboflow export 생성 중...')
    time.sleep(5)
if not export_link:
    raise TimeoutError('5분 안에 export가 준비되지 않았습니다.')

archive = WORKDIR / 'dataset.zip'
urllib.request.urlretrieve(export_link, archive)
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir()
with zipfile.ZipFile(archive) as zip_file:
    zip_file.extractall(DATASET_DIR)
print('Downloaded:', DATASET_DIR)

In [ ]:
# 5. data.yaml 경로 보정 및 Pose 라벨 검사
from collections import Counter
import yaml

data_yaml = DATASET_DIR / 'data.yaml'
config = yaml.safe_load(data_yaml.read_text())
config['path'] = str(DATASET_DIR)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
data_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

kpt_shape = config.get('kpt_shape')
if kpt_shape != [12, 3]:
    raise ValueError(f'예상한 kpt_shape [12, 3]과 다릅니다: {kpt_shape}')
expected_fields = 5 + kpt_shape[0] * kpt_shape[1]
field_counts = Counter()
class_counts = Counter()
for label_path in DATASET_DIR.glob('*/labels/*.txt'):
    for line in label_path.read_text().splitlines():
        if line.strip():
            fields = line.split()
            field_counts[len(fields)] += 1
            class_counts[int(fields[0])] += 1
if set(field_counts) != {expected_fields}:
    raise ValueError(f'비정상 라벨 필드 수: {dict(field_counts)}')
print('classes:', config['names'])
print('kpt_shape:', kpt_shape)
print('objects by class:', dict(class_counts))
print('label fields:', dict(field_counts))

In [ ]:
# 6. YOLO11n Pose 학습
from ultralytics import YOLO

EPOCHS = 100
BATCH = 16  # T4 메모리 부족 시 8로 낮추세요.
IMGSZ = 640
RUN_NAME = 'caterpillar_fall_pose'

model = YOLO('yolo11n-pose.pt')
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    patience=20,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    seed=42,
    deterministic=True,
    amp=True,
    plots=True,
)

In [ ]:
# 7. best.pt를 test split에서 최종 검증
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(BEST_WEIGHTS)
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(data_yaml),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME + '_test',
)
print('Best weights:', BEST_WEIGHTS)

## 이미지 또는 영상 테스트

Colab은 로컬 OpenCV 웹캠 창을 직접 열 수 없으므로 이미지나 영상을 업로드해 테스트합니다.

In [ ]:
# 8. 테스트 이미지/영상 업로드 및 추론
from google.colab import files
from IPython.display import Image, Video, display

uploaded = files.upload()
if not uploaded:
    raise ValueError('업로드된 파일이 없습니다.')
source_path = Path('/content') / next(iter(uploaded))
predict_results = best_model.predict(
    source=str(source_path),
    conf=0.25,
    iou=0.5,
    imgsz=640,
    device=0,
    save=True,
    project=str(RUNS_DIR),
    name='uploaded_test',
)
result_file = RUNS_DIR / 'uploaded_test' / source_path.name
print('Result:', result_file)
if source_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
    display(Image(filename=str(result_file)))
elif result_file.is_file():
    display(Video(str(result_file), embed=True))

In [ ]:
# 9. 학습 모델과 결과 ZIP 다운로드
from google.colab import files

archive_base = Path('/content/caterpillar_fall_pose_results')
archive_path = shutil.make_archive(
    str(archive_base), 'zip', RUNS_DIR / RUN_NAME
)
print('best.pt:', BEST_WEIGHTS)
files.download(str(BEST_WEIGHTS))
files.download(archive_path)